In [2]:
import sys
import os

# SPARK_HOME path
os.environ["SPARK_HOME"] = "/home/fragkiska/spark"

# Add pyspark to Python path
sys.path.append("/home/fragkiska/spark/python")
sys.path.append("/home/fragkiska/spark/python/lib/py4j-0.10.9.7-src.zip")


import pyspark
from pyspark.sql import SparkSession

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = (
    SparkSession.builder
    .appName("Query2-AdvancedDBs")
    .config("spark.executor.instances", "4")
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "2g")
    .getOrCreate()
)


your 131072x1 screen size is bogus. expect trouble
25/12/08 16:32:12 WARN Utils: Your hostname, LAPTOP-POBVNKJ0 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/12/08 16:32:12 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/08 16:32:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/12/08 16:32:13 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [4]:
from pathlib import Path

project_root = Path.cwd()

data_dir = project_root / "data"

crime_data_2010_2019 = data_dir / "LA_Crime_Data_2010_2019.csv"
crime_data_2020_2025 = data_dir / "LA_Crime_Data_2020_2025.csv"
re_codes = data_dir / "RE_codes.csv"

re_codes = (
    spark.read.csv(str(re_codes), header=True, inferSchema=True)
    .withColumnRenamed("Vict Descent", "descent_code")
    .withColumnRenamed("Vict Descent Full", "descent_full")
)

df1 = spark.read.csv(str(crime_data_2010_2019), header=True, inferSchema=True)
df2 = spark.read.csv(str(crime_data_2020_2025), header=True, inferSchema=True)

crime_data = df1.unionByName(df2)

crime_data = (
    crime_data
    .join(re_codes, crime_data["Vict Descent"] == re_codes["descent_code"], "left")
)

crime_data = crime_data.withColumnRenamed("descent_full", "Vict Descent Full")

crime_data = crime_data.drop("descent_code")


crime_data.show(5)   

25/12/08 16:32:28 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+---------+--------------------+--------------------+--------+----+---------+-----------+--------+------+--------------------+--------------+--------+--------+------------+---------+--------------------+--------------+--------------------+------+------------+--------+--------+--------+--------+--------------------+--------------------+-------+---------+--------------------+
|    DR_NO|           Date Rptd|            DATE OCC|TIME OCC|AREA|AREA NAME|Rpt Dist No|Part 1-2|Crm Cd|         Crm Cd Desc|       Mocodes|Vict Age|Vict Sex|Vict Descent|Premis Cd|         Premis Desc|Weapon Used Cd|         Weapon Desc|Status| Status Desc|Crm Cd 1|Crm Cd 2|Crm Cd 3|Crm Cd 4|            LOCATION|        Cross Street|    LAT|      LON|   Vict Descent Full|
+---------+--------------------+--------------------+--------+----+---------+-----------+--------+------+--------------------+--------------+--------+--------+------------+---------+--------------------+--------------+--------------------+------+

In [7]:
import time
from pyspark.sql.functions import col, year, count, desc, round

start_df = time.time()
from pyspark.sql.functions import col, regexp_extract, count, sum, round, desc, dense_rank
from pyspark.sql.window import Window

# extract year
df_year = crime_data.withColumn(
    "year",
    regexp_extract(col("DATE OCC"), r"^(\d{4})", 1).cast("int")
)

# count per descent per year
group_counts = (
    df_year.groupBy("year", "Vict Descent Full")
           .agg(count("*").alias("count"))
           .filter(col("Vict Descent Full").isNotNull())        # <-- ignore NULL
)

# totals per year
year_totals = (
    group_counts.groupBy("year")
                .agg(sum("count").alias("total_year"))
)

# join + percentage
with_percent = (
    group_counts.join(year_totals, on="year")
                .withColumn("percentage", round(col("count") / col("total_year") * 100, 1))
)

# correct ranking AFTER removing NULL
w = Window.partitionBy("year").orderBy(desc("count"))

top3_df = (
    with_percent.withColumn("rank", dense_rank().over(w))
                .filter(col("rank") <= 3)
                .orderBy(col("year").desc(), col("rank").asc())
)

top3_df.show(50, truncate=False)

end_df = time.time()
print(f"DataFrame Implementation Time: {end_df - start_df:.2f} sec")


+----+----------------------+-----+----------+----------+----+
|year|Vict Descent Full     |count|total_year|percentage|rank|
+----+----------------------+-----+----------+----------+----+
|2025|Hispanic/Latin/Mexican|34   |84        |40.5      |1   |
|2025|Unknown               |24   |84        |28.6      |2   |
|2025|White                 |13   |84        |15.5      |3   |
|2024|Hispanic/Latin/Mexican|28576|98363     |29.1      |1   |
|2024|White                 |22958|98363     |23.3      |2   |
|2024|Unknown               |19984|98363     |20.3      |3   |
|2023|Hispanic/Latin/Mexican|69401|200846    |34.6      |1   |
|2023|White                 |44615|200846    |22.2      |2   |
|2023|Black                 |30504|200846    |15.2      |3   |
|2022|Hispanic/Latin/Mexican|73111|205164    |35.6      |1   |
|2022|White                 |46695|205164    |22.8      |2   |
|2022|Black                 |34634|205164    |16.9      |3   |
|2021|Hispanic/Latin/Mexican|63676|181518    |35.1     

In [6]:
df_year.createOrReplaceTempView("crime")
start_sql = time.time()

query = """
WITH base AS (
    SELECT
        CAST(substring(`DATE OCC`, 1, 4) AS INT) AS year,
        `Vict Descent Full` AS descent
    FROM crime
    WHERE `Vict Descent Full` IS NOT NULL AND `Vict Descent Full` != ''
),
group_counts AS (
    SELECT year, descent, COUNT(*) AS count
    FROM base
    GROUP BY year, descent
),
year_totals AS (
    SELECT year, SUM(count) AS total_year
    FROM group_counts
    GROUP BY year
),
with_percent AS (
    SELECT 
        g.year,
        g.descent,
        g.count,
        ROUND(g.count / t.total_year * 100, 1) AS percentage
    FROM group_counts g
    JOIN year_totals t 
        ON g.year = t.year
),
ranked AS (
    SELECT *,
        DENSE_RANK() OVER (PARTITION BY year ORDER BY count DESC) AS rank
    FROM with_percent
)
SELECT 
    year,
    descent AS `Victim Descent`,
    count AS `#`,
    percentage AS `%`
FROM ranked
WHERE rank <= 3
ORDER BY year DESC, count DESC;
"""

top3_sql = spark.sql(query)
top3_sql.show(50, truncate=False)

end_sql = time.time()
print(f"SQL Implementation Time: {end_sql - start_sql:.2f} sec")

+----+----------------------+-----+----+
|year|Victim Descent        |#    |%   |
+----+----------------------+-----+----+
|2025|Hispanic/Latin/Mexican|34   |40.5|
|2025|Unknown               |24   |28.6|
|2025|White                 |13   |15.5|
|2024|Hispanic/Latin/Mexican|28576|29.1|
|2024|White                 |22958|23.3|
|2024|Unknown               |19984|20.3|
|2023|Hispanic/Latin/Mexican|69401|34.6|
|2023|White                 |44615|22.2|
|2023|Black                 |30504|15.2|
|2022|Hispanic/Latin/Mexican|73111|35.6|
|2022|White                 |46695|22.8|
|2022|Black                 |34634|16.9|
|2021|Hispanic/Latin/Mexican|63676|35.1|
|2021|White                 |44523|24.5|
|2021|Black                 |30173|16.6|
|2020|Hispanic/Latin/Mexican|61606|35.3|
|2020|White                 |42638|24.5|
|2020|Black                 |28785|16.5|
|2019|Hispanic/Latin/Mexican|72458|36.4|
|2019|White                 |48863|24.5|
|2019|Black                 |33157|16.6|
|2018|Hispanic/L